# Model o'g'irlash: API orqali og'irliklarni tiklash

Bir kompaniya yillar davomida ma'lumot yig'ib, model o'qitgan. Model ularning serverida turadi, tashqariga faqat
API ochiq: siz kirish yuborasiz, u ehtimollik qaytaradi. Kodni ham, og'irliklarni ham hech kim ko'rsatmaydi.

Savol: shu API'ning o'zi bilan modelning aynan nusxasini olsa bo'ladimi? Bo'ladi. Bu darsda buni atigi o'ntacha
so'rov bilan qilasiz. Bunday hujum **model extraction** (model o'g'irlash) deyiladi.

Bu darsda uch narsani o'rganasiz:

1. Chiziqli modelning ichida nima borligini: og'irliklar va bias.
2. Ehtimollikdan modelning «xom» hisobini qanday qaytarib olishni.
3. Har bir so'rovni bitta noma'lumni ochadigan qilib tanlashni.

Python'da ro'yxat va `for` siklini, maktab algebrasidan esa bir noma'lumli tenglamani bilsangiz yetarli.

Bir gap: bu o'quv muhiti. Birovning modelini ruxsatsiz ko'chirish qonunni ham, xizmat shartlarini ham buzadi.

## 1. Labga ulanamiz

Platformadagi lab sahifasini oching. U yerda **Token** degan maydon bor: undagi satrni nusxalab, pastdagi
`TOKEN` ichiga qo'ying. Token faqat sizga tegishli va bir necha soatdan keyin eskiradi. Eskirib qolsa, lab
sahifasini yangilab, yangisini oling.

In [ ]:
BASE = "https://platform.hacknow.uz/ai-poligon"
TOKEN = ""   # lab sahifasidagi Token maydonidan nusxalab, qo'shtirnoq ichiga qo'ying

assert TOKEN, "Avval TOKEN ni to'ldiring: platformadagi lab sahifasidan nusxalang."

import math
import urllib.request

urllib.request.urlretrieve(BASE + "/client/hn_lab.py", "hn_lab.py")   # labning kichik mijozi
from hn_lab import HnLab

lab = HnLab(BASE, "ml-model-ogirlash", TOKEN)
info = lab.task()
print(info["lab"]["title"])
print("Urinish:", info["attempt"], "| sarflangan so'rov:", info["queries_used"], "/", info["query_budget"])
if info["solved"]:
    print("Bu lab sizda allaqachon yechilgan. Flag:", info.get("flag"))
if info["query_budget"] - info["queries_used"] < 10:
    print("Diqqat: so'rovlar kam qoldi. Yetmay qolsa, lab.reset() bilan yangi urinish boshlang va notebook'ni boshidan yurgizing.")

## 2. Model ichida nima bor?

Model oltita son qabul qiladi. Har bir kirishning o'z **og'irligi** bor. Model har kirishni o'z og'irligiga
ko'paytiradi, hammasini qo'shadi va ustiga yana bitta son, **bias**ni qo'shadi:

`yig'indi = w1·x1 + w2·x2 + ... + w6·x6 + b`

Keyin bu yig'indini 0 bilan 1 orasiga «siqadi» va natijani ehtimollik sifatida qaytaradi. Demak, butun model
yettita sondan iborat: oltita og'irlik va bitta bias. Vazifangiz shu yettita sonni topish.

In [ ]:
vazifa = info["task"]
n = vazifa["n_features"]

print("Kirishlar soni:", n, "| har kirish shu oraliqda bo'lishi kerak:", vazifa["input_range"])
print("Ehtimollik nechta xonagacha yaxlitlanadi:", vazifa["proba_decimals"])
print("Ruxsat etilgan xato (har bir son uchun):", vazifa["tolerance"])

Javobingiz qabul qilinishi uchun topgan har bir soningiz haqiqiysidan ko'pi bilan 0.1 ga farq qilishi kerak.

So'rovlar cheklangan: bitta urinishda 60 ta. Har yuborilgan qator bitta so'rov, javobni topshirish ham bitta
so'rov hisoblanadi. So'rovlar tugab qolsa, `lab.reset()` yangi urinish boshlaydi. Lekin ehtiyot bo'ling:
reset'dan keyin model ham yangilanadi, ya'ni hammasini boshidan o'lchash kerak bo'ladi.

**Siqishni orqaga qaytarish.** Model yig'indini siqib ehtimollikka aylantiradi. Bu amalni teskarisiga
bajarsa bo'ladi: `logit` funksiyasi ehtimollikdan yig'indini qayta chiqarib beradi. Funksiya tayyor turibdi,
formulasini yodlash shart emas.

In [ ]:
def logit(ehtimollik):
    ehtimollik = min(max(ehtimollik, 0.0001), 0.9999)   # 0 yoki 1 chiqib qolsa, cheksizlikka ketib qolmasin
    return math.log(ehtimollik / (1 - ehtimollik))


def sigmoid(yigindi):
    return 1 / (1 + math.exp(-yigindi))

## 3. Avval bias'ni topamiz

Formulaga yana bir qarang. Hamma kirishni nol qilib yuborsak, ko'paytmalarning hammasi nolga aylanadi va
yig'indida faqat bias qoladi. Demak, bitta so'rov bilan bias'ni bilib olamiz.

In [ ]:
nollar = [0.0] * n
javob = lab.score({"rows": [nollar]})
p0 = javob["output"]["proba"][0]
b = logit(p0)

print("ehtimollik:", p0)
print("bias taxminan:", round(b, 3))
print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])

## 4. Birinchi og'irlikni topamiz

Endi faqat birinchi kirishni `h` ga teng qilamiz, qolganlari nol bo'lib turaveradi. Shunda yig'indi
`w1·h + b` ga teng bo'ladi. Bias'ni bilamiz, `h` ni o'zimiz tanladik, demak `w1` ni topish oson:

`w1 = (yig'indi - b) / h`

In [ ]:
h = 0.5

qator = [0.0] * n
qator[0] = h
p = lab.score({"rows": [qator]})["output"]["proba"][0]
w_birinchi = (logit(p) - b) / h

print("ehtimollik:", p)
print("birinchi og'irlik taxminan:", round(w_birinchi, 3))

Nega aynan 0.5? Qadam juda kichik bo'lsa, ehtimollik deyarli o'zgarmaydi va yaxlitlash xatosi natijani buzadi.
Juda katta ham ololmaysiz: kirish -1 bilan 1 orasida bo'lishi shart. 0.5 shu ikkisining o'rtasidagi tanlov.

## 5. Topshiriq 1: qolgan og'irliklar uchun so'rovlar tuzing

Xuddi shu ishni oltita kirishning har biri uchun qilasiz. Pastdagi katakka `for` sikli yozing: har qadamda
nollardan iborat yangi qator oling, navbatdagi kirishga `h` ni qo'ying va qatorni `qatorlar` ro'yxatiga qo'shing.
Oxirida ro'yxatda oltita qator bo'lishi kerak: birinchisida `h` birinchi o'rinda, ikkinchisida ikkinchi o'rinda
va hokazo.

In [ ]:
qatorlar = []   # oltita qator: i-qatorda faqat i-kirish h ga teng, qolganlari nol

# TOPSHIRIQ 1: siklni shu yerga yozing.

Keyingi katak ro'yxatingizni tekshiradi va hamma qatorni bitta chaqiruvda yuboradi. Javoblar `ehtimollar`
ro'yxatiga tushadi: birinchi ehtimollik birinchi qatorga, ikkinchisi ikkinchi qatorga tegishli.

In [ ]:
def qatorlar_togri(royxat):
    if len(royxat) != n:
        print("Hali tayyor emas: ro'yxatda", len(royxat), "ta qator bor, kerak:", n)
        return False
    for i, satr in enumerate(royxat):
        if len(satr) != n or satr[i] != h or any(satr[k] != 0.0 for k in range(n) if k != i):
            print("Bu qatorda xato bor (tartib raqami %d): unda faqat shu o'rindagi kirish h ga teng, qolganlari nol bo'lishi kerak." % i)
            return False
    return True


ehtimollar = []
if qatorlar_togri(qatorlar):
    javob = lab.score({"rows": qatorlar})
    ehtimollar = javob["output"]["proba"]
    print("ehtimollar:", ehtimollar)
    print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])

## 6. Topshiriq 2: og'irliklarni hisoblang

Endi har bir ehtimollikdan o'z og'irligini chiqaring. Formula 4-bo'limdagi bilan bir xil:
ehtimollikni `logit` bilan yig'indiga aylantirasiz, undan `b` ni ayirasiz va `h` ga bo'lasiz. Natijalarni
`vaznlar` ro'yxatiga, `ehtimollar` dagi tartibda qo'shing.

In [ ]:
vaznlar = []   # oltita og'irlik, kirishlar tartibida

# TOPSHIRIQ 2: hisobni shu yerga yozing.

## 7. Nusxani tekshiramiz va topshiramiz

Topshirishdan oldin nusxamiz haqiqiy modelga o'xshaydimi, bilib olamiz. Buning uchun hali ishlatmagan bitta
nuqtani olamiz, javobni avval o'zimiz hisoblaymiz, keyin API'dan so'raymiz va ikkalasini solishtiramiz.
Farq juda kichik bo'lsa, nusxa to'g'ri chiqqan.

In [ ]:
if len(vaznlar) != n:
    print("Hali tayyor emas: Topshiriq 2 ni bajaring (vaznlar ichida", len(vaznlar), "ta son bor, kerak:", n, ").")
else:
    sinov_nuqta = [0.3, -0.2, 0.1, 0.4, -0.5, 0.2]
    bizniki = sigmoid(sum(w * x for w, x in zip(vaznlar, sinov_nuqta)) + b)
    haqiqiy = lab.score({"rows": [sinov_nuqta]})["output"]["proba"][0]
    print("bizning nusxa: %.3f | haqiqiy model: %.3f" % (bizniki, haqiqiy))
    if abs(bizniki - haqiqiy) > 0.02:
        print("Farq katta: hisobda xato bor. Topshiriq 2 dagi formulani 4-bo'lim bilan solishtiring.")
    else:
        javob = lab.score({"weights": vaznlar, "bias": b})
        print(javob["output"], "—", javob["message"])
        if javob["solved"]:
            print("Bo'ldi! Model ko'chirildi.")
            print("Flag:", javob["flag"])

Flag chiqqan bo'lsa, uni platformadagi topshiriq sahifasiga yuboring. Flag faqat sizniki: boshqa odamning
flag'i sizda ishlamaydi.

## 8. Nima bo'ldi?

Kompaniya modelni yashirgan edi, siz esa uni o'ntacha so'rov bilan ko'chirib oldingiz. Buning ikki sababi bor.
Birinchisi, model chiziqli: ichida bor-yo'g'i yettita son. Ikkinchisi, API ehtimollikni aniq qaytaradi, undan
esa modelning xom hisobini qayta tiklasa bo'ladi. Ehtimollikni uch xonagacha yaxlitlash ham to'sqinlik qilmadi.

Haqiqiy modellar ancha katta va ularni bitta formula bilan ochib bo'lmaydi. Lekin g'oya o'sha-o'sha: hujumchi
juda ko'p so'rov yuboradi, javoblarni yig'adi va shu javoblar ustida o'z modelini o'qitadi. Natijada asl modelga
juda o'xshash nusxa hosil bo'ladi, uni yaratish uchun sarflangan vaqt va pul esa hujumchiga tekinga tushadi.

Himoyachi nima qilishi mumkin? Ehtimollik o'rniga faqat yakuniy javobni qaytarish hujumni qimmatlashtiradi,
lekin to'xtatmaydi. So'rovlar sonini cheklash va g'alati so'rovlarni kuzatish foyda beradi: oddiy mijoz hech
qachon «hamma kirish nol» yoki «faqat bitta kirish 0.5» degan so'rov yubormaydi. Javobga ozgina shovqin qo'shish
ham mumkin, lekin bu halol mijozlar uchun ham aniqlikni pasaytiradi. Shuning uchun amalda bir nechta chora
birga ishlatiladi.

## Qo'shimcha (xohlasangiz)

1. `h` ni 0.05 qilib ko'ring. Topilgan og'irliklar qanday o'zgaradi va nega?
2. API ehtimollikni uch xona emas, bir xonagacha yaxlitlasa, bu usul ishlarmidi? Uni qanday tuzatgan bo'lardingiz?
3. Yettitadan kam so'rov bilan uddalasa bo'ladimi? Nega?